# Neural Network Preprocessing - OOF RankGauss Normalization

Implements Out-of-Fold (OOF) QuantileTransformer scaling to prevent data leakage.

**Key Features:**
- Each training sample is scaled using a transformer fit only on other folds
- Test set scaled using transformer fit on entire training set
- Median imputation before scaling

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().absolute().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from config_loader import get_path, get_seed

RANDOM_STATE = get_seed()
DATA_DIR = get_path('data_processed')

TRAIN_PATH = DATA_DIR / 'further_train_features.parquet'
TEST_PATH = DATA_DIR / 'further_test_features.parquet'
FOLDS_PATH = DATA_DIR / 'train_folds.csv'

OUTPUT_TRAIN = DATA_DIR / 'further_train_processed_nn.parquet'
OUTPUT_TEST = DATA_DIR / 'further_test_processed_nn.parquet'

print(f'Random State: {RANDOM_STATE}')
print(f'Train Path: {TRAIN_PATH}')
print(f'Test Path: {TEST_PATH}')

Random State: 15
Train Path: d:\MALLORN Private\data\processed\further_train_features.parquet
Test Path: d:\MALLORN Private\data\processed\further_test_features.parquet


In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import QuantileTransformer
import warnings
warnings.filterwarnings('ignore')

In [3]:
print('Loading data...')
train = pd.read_parquet(TRAIN_PATH)
test = pd.read_parquet(TEST_PATH)
folds = pd.read_csv(FOLDS_PATH)

train = train.merge(folds[['object_id', 'kfold']], on='object_id', how='left')

print(f'Train shape: {train.shape}')
print(f'Test shape: {test.shape}')
print(f'Folds distribution: {train["kfold"].value_counts().sort_index().to_dict()}')

Loading data...
Train shape: (3043, 347)
Test shape: (7135, 345)
Folds distribution: {0: 609, 1: 609, 2: 609, 3: 608, 4: 608}


In [4]:
meta_cols = ['object_id', 'target', 'split', 'SpecType', 'kfold']
feature_cols = [c for c in train.columns if c not in meta_cols]
print(f'Number of features: {len(feature_cols)}')

train_meta = train[['object_id', 'target', 'kfold']].copy()
test_meta = test[['object_id']].copy()

X_train_raw = train[feature_cols].copy()
X_test_raw = test[feature_cols].copy()

Number of features: 344


In [5]:
print('Imputing NaN with median from full train set...')
train_medians = X_train_raw.median()

nan_before_train = X_train_raw.isna().sum().sum()
nan_before_test = X_test_raw.isna().sum().sum()

X_train_imputed = X_train_raw.fillna(train_medians).astype(np.float32)
X_test_imputed = X_test_raw.fillna(train_medians).astype(np.float32)

print(f'Train NaN: {nan_before_train} -> {X_train_imputed.isna().sum().sum()}')
print(f'Test NaN: {nan_before_test} -> {X_test_imputed.isna().sum().sum()}')
print(f'Dtype: {X_train_imputed.dtypes.iloc[0]}')

Imputing NaN with median from full train set...
Train NaN: 8192 -> 0
Test NaN: 19245 -> 0
Dtype: float32


In [6]:
print('\nApplying OOF RankGauss scaling to training set...')
kfold = train['kfold'].values
X_train_scaled = X_train_imputed.copy()

for f in range(5):
    train_idx = kfold != f
    val_idx = kfold == f
    
    scaler = QuantileTransformer(output_distribution='normal', random_state=RANDOM_STATE)
    scaler.fit(X_train_imputed.loc[train_idx])
    
    X_train_scaled.loc[val_idx] = scaler.transform(X_train_imputed.loc[val_idx])
    
    print(f'  Fold {f}: fit on {train_idx.sum()} samples, transform {val_idx.sum()} samples')

print('OOF scaling complete.')


Applying OOF RankGauss scaling to training set...
  Fold 0: fit on 2434 samples, transform 609 samples
  Fold 1: fit on 2434 samples, transform 609 samples
  Fold 2: fit on 2434 samples, transform 609 samples
  Fold 3: fit on 2435 samples, transform 608 samples
  Fold 4: fit on 2435 samples, transform 608 samples
OOF scaling complete.


In [7]:
print('\nScaling test set using full training data...')
final_scaler = QuantileTransformer(output_distribution='normal', random_state=RANDOM_STATE)
final_scaler.fit(X_train_imputed)
X_test_scaled = pd.DataFrame(
    final_scaler.transform(X_test_imputed),
    columns=feature_cols,
    index=X_test_imputed.index
)
print('Test scaling complete.')


Scaling test set using full training data...
Test scaling complete.


In [8]:
train_final = pd.concat([train_meta, X_train_scaled], axis=1)
test_final = pd.concat([test_meta, X_test_scaled], axis=1)

print(f'\nFinal train shape: {train_final.shape}')
print(f'Final test shape: {test_final.shape}')

train_final.drop(columns=['kfold'], inplace=True)
test_final.drop(columns=['kfold'], inplace=True, errors='ignore')

train_final.to_parquet(OUTPUT_TRAIN, index=False)
test_final.to_parquet(OUTPUT_TEST, index=False)

print(f'\nSaved: {OUTPUT_TRAIN}')
print(f'Saved: {OUTPUT_TEST}')


Final train shape: (3043, 347)
Final test shape: (7135, 345)

Saved: d:\MALLORN Private\data\processed\further_train_processed_nn.parquet
Saved: d:\MALLORN Private\data\processed\further_test_processed_nn.parquet


In [9]:
print('\n=== Verification ===')
print(f'Train scaled stats (sample feature):')
sample_col = feature_cols[0]
print(f'  {sample_col}: mean={X_train_scaled[sample_col].mean():.4f}, std={X_train_scaled[sample_col].std():.4f}')
print(f'Test scaled stats (sample feature):')
print(f'  {sample_col}: mean={X_test_scaled[sample_col].mean():.4f}, std={X_test_scaled[sample_col].std():.4f}')


=== Verification ===
Train scaled stats (sample feature):
  Flux_mean_g: mean=-0.0001, std=1.0076
Test scaled stats (sample feature):
  Flux_mean_g: mean=0.0046, std=1.0018
